# Heart Disease Prediction

This notebook follows the project workflow documented in the README: data loading, exploration, preparation, model selection, optimization, and evaluation. Run the sections from top to bottom.

## 1. Dataset
Load the dataset and inspect its first rows and schema.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore') 
import gc

In [ ]:
file = pd.read_csv('heart.csv')
file1 = file.copy()

In [ ]:
file.head()

In [ ]:
file.info()
 


## 2. Exploratory Data Analysis (EDA)
Explore the distributions of the key numerical features before preprocessing.

In [ ]:
def plot(var,num):
  plt.subplot(2,2,num)
  sns.histplot(file[var],kde=True)
  plt.tight_layout()
plot('Age',1)
plot('RestingBP',2)
plot('Cholesterol',3)
plot('MaxHR',4)

## 3. Preprocessing
Remove duplicates, replace invalid zero values, encode categorical variables, and select the modelling features.

In [ ]:
file = file.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', file.shape)

In [ ]:
new_mean = file.loc[file['Cholesterol'] != 0, 'Cholesterol'].mean()
print(round(new_mean, 2))

In [ ]:
file['Cholesterol'] = file['Cholesterol'].astype(float)
file['Cholesterol'] = file['Cholesterol'].replace(0, round(new_mean, 2))

In [ ]:
file['RestingBP'] = file['RestingBP'].astype(float)
resting_bp_mean = file.loc[file['RestingBP'] != 0, 'RestingBP'].mean()
file.loc[file['RestingBP'] == 0, 'RestingBP'] = round(resting_bp_mean, 2)

#preprocessing


In [ ]:
file['Cholesterol'].value_counts()


In [ ]:
file['MaxHR'].value_counts()

In [ ]:
file['RestingBP'].value_counts()

In [ ]:
file['Age'].value_counts()

In [ ]:
file['ChestPainType'].value_counts()

In [ ]:
file['Sex'].value_counts()

In [ ]:
file.info()

In [ ]:
file['Sex'].value_counts()

In [ ]:
file =pd.get_dummies(file,columns=['ChestPainType'],drop_first=True,dtype=int)

In [ ]:
file['ST_Slope'].value_counts()

In [ ]:
file=pd.get_dummies(file,columns=['ExerciseAngina'],drop_first=True,dtype=int)

In [ ]:
file=pd.get_dummies(file,columns=['RestingECG'],drop_first=True,dtype=int)

In [ ]:
file.head()

In [ ]:
file.info()

In [ ]:
file = pd.get_dummies(file, columns=['ST_Slope'], drop_first=True, dtype=int)

In [ ]:
file.head()

In [ ]:
file.info()

In [ ]:
file = pd.get_dummies(file, columns=['Sex'], drop_first=True, dtype=int)

In [ ]:
file['Oldpeak'].value_counts()

In [ ]:
del file1
gc.collect()

In [ ]:
file = file.astype({
    'Cholesterol': int,
    'RestingBP': int
})

In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.10
features = [
    'ChestPainType_ATA',
    'ChestPainType_NAP',
    'ChestPainType_TA',
    'ExerciseAngina_Y',
    'RestingECG_Normal',
    'RestingECG_ST',
    'ST_Slope_Flat',
    'ST_Slope_Up',
    'Sex_M'
]

results = {}
for feature in features:
    contingency_table = pd.crosstab(file[feature], file['HeartDisease'])
    chi2_stat, p_val, df, expected = chi2_contingency(contingency_table)
    decision = 'keep feature (null hypothesis rejected)' if p_val < alpha else 'discard feature (null hypothesis accepted)'
    results[feature] = {
        'chi2_stat': chi2_stat,
        'p_value': p_val,
        'degree_of_freedom': df,
        'decision': decision
    }

results = pd.DataFrame(results).T
print(results)

In [ ]:
selected_features = [
    'Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak',
    'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA',
    'ExerciseAngina_Y', 'RestingECG_Normal', 'RestingECG_ST',
    'ST_Slope_Flat', 'ST_Slope_Up', 'Sex_M', 'HeartDisease'
]

cleaned = file[selected_features].copy()
cleaned.head()

In [ ]:
from scipy.stats import pearsonr

numeric_features = [
    'Age', 'RestingBP', 'Cholesterol',
    'FastingBS', 'MaxHR', 'Oldpeak'
]

correlation = {
    feature: pearsonr(cleaned[feature], cleaned['HeartDisease'])[0]
    for feature in numeric_features
}

In [ ]:
correlation = pd.DataFrame(list(correlation.items()), columns=['feature', 'correlation'])

In [ ]:
print(correlation)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(correlation['feature'], correlation['correlation'], edgecolor='black')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
print('done')

## 4. Train/Test Split
Separate predictors and target, then create stratified training and test sets.

In [ ]:
X = cleaned.drop('HeartDisease',axis=1)
y= cleaned['HeartDisease']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print(X_test)

## 5. Pipeline
Build a reusable pipeline that standardizes features and fits logistic regression without data leakage.

In [ ]:
from sklearn.pipeline  import Pipeline
pipeline = Pipeline([
  ('scaler',StandardScaler()),
  ('logreg',LogisticRegression())
])

## 6. GridSearchCV
Tune the logistic-regression regularization strength (C) using recall as the selection metric.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "logreg__C":[0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="recall"
)

grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:", grid.best_params_)
print("Best CV recall:", grid.best_score_)

In [ ]:
 
grid1=GridSearchCV(pipeline,param_grid={'logreg__C':[0.0001, 0.001, 0.01, 0.1, 1, 10, 100]} ,
cv=5,scoring='recall')
grid1.fit(X_train,y_train)
print('best parametrs',grid1.best_params_)
print('best estimator',grid1.best_estimator_)
print('best score ',grid1.best_score_)

In [ ]:
print("\nAll configurations:")

for params, score in zip(
    grid1.cv_results_["params"],
    grid1.cv_results_["mean_test_score"]
):
    print(params, score)


## 7. Cross-validation
Measure the pipeline's consistency with recall-focused and multi-metric cross-validation.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
  pipeline,
  X_train,
  y_train,
  cv=10,
  scoring='recall'

)

In [ ]:
print("the mean of the scores is :", scores.mean())
print("the std of the scores is : ",scores.std())

In [ ]:
from sklearn.model_selection import cross_validate

results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

for metric in ["test_accuracy", "test_precision", "test_recall",
               "test_f1", "test_roc_auc"]:

    print(metric)
    print("Mean:", results[metric].mean())
    print("Std :", results[metric].std())
    print()

## 8. Best Logistic Regression
Select the best estimator found by GridSearchCV and generate class predictions and positive-class probabilities for the held-out test set.

In [ ]:
best_model = grid1.best_estimator_
print('Selected model:', best_model)

y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]


## 9. Threshold Optimization
Compare classification thresholds to understand the trade-off among recall, precision, F1 score, and accuracy.

In [ ]:
nod = np.linspace(0.10,0.50,num=15,endpoint=True,retstep=False)
for threshold in nod:
  y_pred_threshold=(y_pred_proba>=threshold).astype(int)
  recall=metrics.recall_score(y_test,y_pred_threshold)
  accuracy=metrics.accuracy_score(y_test,y_pred_threshold)
  precision = metrics.precision_score(y_test,y_pred_threshold)
  f1 = metrics.f1_score(y_test,y_pred_threshold)
  print(
    f"threshhold is: {threshold:1f}|"
   f"accuracy is : {accuracy:.2f}|"
    f"recall: {recall:.3f}|"
    f"precision: {precision:.3f}|"
    f"F1 is : {f1:3f}"
  )


## 10. Final Evaluation
Report the tuned model's test-set metrics and confusion matrix using the default classification threshold.

In [ ]:
from sklearn import metrics
print("the acurracy is ",metrics.accuracy_score(y_test,y_pred))
print("the F1 score is ",metrics.f1_score(y_test,y_pred))
print("the precision_score is ",metrics.precision_score(y_test,y_pred))
print("the recall_score is ",metrics.recall_score(y_test,y_pred))
print("the roc_auc_score is ",metrics.roc_auc_score(y_test,y_pred_proba))

In [ ]:
cm = metrics.confusion_matrix(y_test,y_pred)
print(cm)